# IEP-1001 CASE 3 — RAGAS 재측정 (Solar judge)

> **목적**: 배포 LLM인 Solar(`solar-pro3`)로 Faithfulness + Answer Relevancy 재측정  
> **배경**: 기존 수치는 `llama3.1:8b` judge 기준 — 배포 LLM과 judge를 일치시켜 신뢰도 확보  
> **컬렉션**: `ipcc_1001_case3_v1` (506개 청크, chunk_size=1000, overlap=200)  
> **Answer 생성 + Judge LLM**: `solar-pro3`  
> **Answer Relevancy 임베딩**: `jhgan/ko-sroberta-multitask` (Solar API `n=1` 제약으로 대체)

## 측정 결과 (2026-04-26)

| 지표 | llama3.1:8b | Solar (이번) | NaN |
| :--- | :---: | :---: | :---: |
| Context Recall | 0.8537 | — | — |
| Context Precision | 0.6117 | — | — |
| **Faithfulness** | 0.4361 | **0.2748** | 38개 |
| **Answer Relevancy** | 0.6143 | **0.3409** | 0개 |

> Context Recall/Precision은 LLM 무관 — 재측정 불필요

## ⚠️ Solar RAGAS 호환 이슈

| 이슈 | 원인 | 해결 |
| :--- | :--- | :--- |
| `n must be 1` (Answer Relevancy 전량 실패) | RAGAS 기본 `strictness=3` → `n=3` 요청 / Solar `n=1`만 지원 | `AnswerRelevancy(strictness=1)` |
| `$.input is invalid` (embedding 오류) | Solar embedding이 RAGAS 입력 형식 거부 | `jhgan` 임베딩으로 대체 |

## 실행 순서

```
STEP 1  패키지 설치 → 런타임 자동 재시작
  ↓ 재시작 후 STEP 2부터
STEP 2  API 키 설정
STEP 3  Config 정의
STEP 4  Google Drive 마운트
STEP 5  ChromaDB 패치 + 임베딩 모델 + ChromaDB 로드
STEP 6  골든 데이터셋 로드
STEP 7  Solar로 Answer 생성 (체크포인트 포함)
STEP 8  (선택) 세션 복원
STEP 9  RAGAS 평가 모델 설정
STEP 10 Faithfulness 평가
STEP 11 Answer Relevancy 평가
STEP 12 결과 병합 + 유형별 분석
STEP 13 Solar vs llama 비교
STEP 14 생존 편향 보정 + 최종 저장 + README 마크다운 출력
```

## STEP 1 — 패키지 설치

> ⚠️ 실행 후 런타임이 자동 재시작됩니다. 재시작 후 **STEP 2부터** 실행하세요.
>
> - `numpy==1.26.4` 강제 재설치: Colab 기본 numpy 바이너리 충돌 방지
> - `chromadb==0.5.11` 버전 고정: ChromaDB 패치 코드 검증 버전
> - 나머지 패키지: 버전 고정 없음 → pip 자동 해결

In [ ]:
import subprocess, sys

# numpy 바이너리 충돌 방지 (IEP-4001 방법 B에서 검증된 방식)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'numpy', '-y'], capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy==1.26.4', '--force-reinstall', '-q'], capture_output=True)

!pip install \
    langchain-community \
    langchain-huggingface \
    langchain-openai \
    chromadb==0.5.11 \
    sentence-transformers \
    ragas \
    -q

!pip show openai langchain-openai | grep -E "^(Name|Version)"

print("\u2705 설치 완료 — 런타임 재시작 후 STEP 2부터 실행하세요")

import os
os.kill(os.getpid(), 9)

## STEP 2 — API 키 설정

In [ ]:
import getpass, os

# Colab Secrets(🔑)에 등록한 경우 아래 주석 해제
# from google.colab import userdata
# UPSTAGE_API_KEY = userdata.get('UPSTAGE_API_KEY')

UPSTAGE_API_KEY = getpass.getpass("UPSTAGE_API_KEY 입력: ")
os.environ["UPSTAGE_API_KEY"] = UPSTAGE_API_KEY
print(f"\u2705 API 키 설정 완료 (앞 8자리: {UPSTAGE_API_KEY[:8]}...)")

## STEP 3 — Config 정의

In [ ]:
# ── 경로 ───────────────────────────────────────────────────────────
CHROMA_DIR  = "/content/drive/MyDrive/IPCC_data/IEP_1001_CASE3/chroma_db"
GOLDEN_CSV  = "/content/drive/MyDrive/IPCC_data/IEP_1002/golden_dataset_100.csv"
RESULTS_DIR = "/content/drive/MyDrive/IPCC_data/IEP_1001_CASE3/results_solar"

# ── 모델 ───────────────────────────────────────────────────────────
COLLECTION_NAME = "ipcc_1001_case3_v1"
EMBED_MODEL     = "jhgan/ko-sroberta-multitask"
SOLAR_BASE_URL  = "https://api.upstage.ai/v1"
SOLAR_MODEL     = "solar-pro3"
TOP_K           = 3
EXPECTED_CHUNKS = 506

# ── 저장 경로 ──────────────────────────────────────────────────────
SAVE_RETRIEVED    = f"{RESULTS_DIR}/iep1001_case3_solar_retrieved.csv"
SAVE_FAITHFULNESS = f"{RESULTS_DIR}/iep1001_case3_solar_faithfulness.csv"
SAVE_RELEVANCY    = f"{RESULTS_DIR}/iep1001_case3_solar_relevancy.csv"
SAVE_RAW          = f"{RESULTS_DIR}/iep1001_case3_solar_raw.csv"
SAVE_SUMMARY      = f"{RESULTS_DIR}/iep1001_case3_solar_summary.csv"

# ── 기존 수치 (llama3.1:8b 기준, 비교용) ──────────────────────────
LLAMA_RESULTS = {
    "전체":     {"context_recall": 0.8537, "context_precision": 0.6117, "faithfulness": 0.4361, "answer_relevancy": 0.6143},
    "사실 확인": {"context_recall": 0.7381, "context_precision": 0.9133, "faithfulness": 0.6562, "answer_relevancy": 0.5767},
    "비교":     {"context_recall": 0.8600, "context_precision": 0.6733, "faithfulness": 0.3523, "answer_relevancy": 0.5846},
    "의견/예측": {"context_recall": 0.9757, "context_precision": 0.7800, "faithfulness": 0.3908, "answer_relevancy": 0.6822},
    "범위 밖":  {"context_recall": 0.8264, "context_precision": 0.0800, "faithfulness": 0.3261, "answer_relevancy": 0.6066},
}

print("\u2705 Config 완료")
print(f"   컬렉션   : {COLLECTION_NAME} ({EXPECTED_CHUNKS}개)")
print(f"   Judge LLM: {SOLAR_MODEL}")
print(f"   결과 저장 : {RESULTS_DIR}")

## STEP 4 — Google Drive 마운트

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.exists(CHROMA_DIR), f"\u274c ChromaDB 없음: {CHROMA_DIR}"
assert os.path.exists(GOLDEN_CSV), f"\u274c 골든 데이터셋 없음: {GOLDEN_CSV}"

print("\u2705 Drive 마운트 완료")
print(f"   ChromaDB   : {CHROMA_DIR}")
print(f"   골든 데이터셋: {GOLDEN_CSV}")
print(f"   결과 저장   : {RESULTS_DIR}")

## STEP 5 — ChromaDB 패치 + 임베딩 모델 + ChromaDB 로드

> ChromaDB 구버전으로 생성된 DB는 `config_json_str={}`으로 저장되어 `0.5.11`에서 `KeyError: '_type'` 오류 발생.  
> sqlite3로 직접 패치. 이미 패치된 경우 자동 건너뜀.

In [ ]:
import sqlite3
from chromadb.api.configuration import CollectionConfigurationInternal

sqlite_path  = f"{CHROMA_DIR}/chroma.sqlite3"
FIXED_CONFIG = '{"_type": "CollectionConfigurationInternal", "hnsw_configuration": {"_type": "HNSWConfigurationInternal", "space": "cosine", "ef_construction": 100, "ef_search": 10, "num_threads": 4, "resize_factor": 1.2, "batch_size": 100, "sync_threshold": 1000, "M": 16}}'

CollectionConfigurationInternal.from_json_str(FIXED_CONFIG)  # 포맷 사전 검증

conn    = sqlite3.connect(sqlite_path)
cursor  = conn.cursor()
cursor.execute("SELECT name, config_json_str FROM collections;")
rows    = cursor.fetchall()
patched = 0
for name, config in rows:
    if config == "{}":
        cursor.execute("UPDATE collections SET config_json_str = ? WHERE name = ?", (FIXED_CONFIG, name))
        patched += 1
        print(f"   패치 적용: {name}")
    else:
        print(f"   건너뜀 (이미 패치됨): {name}")
conn.commit()
conn.close()
print(f"\u2705 ChromaDB 패치 {'완료' if patched else '불필요 — 이미 정상'}")

In [ ]:
import warnings, torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"   device: {device}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)
count = vectorstore._collection.count()
assert count == EXPECTED_CHUNKS, f"\u26a0\ufe0f  청크 수 불일치: 실제 {count}개 (예상 {EXPECTED_CHUNKS}개)"
print(f"\u2705 ChromaDB 로드 완료: {COLLECTION_NAME} ({count}개 청크)")

## STEP 6 — 골든 데이터셋 로드

In [ ]:
import pandas as pd

df_gold = pd.read_csv(GOLDEN_CSV)
assert len(df_gold) == 100,             f"\u26a0\ufe0f  골든 데이터셋 {len(df_gold)}개 (100개여야 함)"
assert 'user_input' in df_gold.columns, "\u274c 'user_input' 컬럼 없음"
assert 'reference'  in df_gold.columns, "\u274c 'reference' 컬럼 없음"

print(f"\u2705 골든 데이터셋 로드 완료: {len(df_gold)}개")
print(df_gold['Type'].value_counts().to_string())

## STEP 7 — Solar로 Answer 생성

> - 10개마다 체크포인트 자동 저장 — 세션이 끊겨도 이어서 실행 가능
> - 예상 소요: 약 3~5분 (Solar 약 1.7초/건 × 100건)
> - 이미 완료된 경우 체크포인트에서 자동으로 이어서 실행

In [ ]:
from openai import OpenAI
from tqdm.notebook import tqdm
import time

solar_client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url=SOLAR_BASE_URL
)

# IEP-4001 rag.py 와 동일한 프롬프트
RAG_PROMPT = """당신은 IPCC AR6 기후변화 보고서 전문 안내 AI입니다.
반드시 아래 제공된 문서 내용만을 근거로 답변하세요.
문서에 없는 내용은 추측하지 말고 \"해당 내용은 IPCC 보고서에서 찾을 수 없습니다.\"라고 답하세요.

[참고 문서]
{context}

[질문]
{question}

[답변]"""

retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})


def generate_answer_solar(question: str, retry: int = 2) -> str:
    docs    = retriever.invoke(question)
    context = "\n\n".join(
        f"[청크 {i+1} | 페이지 {doc.metadata.get('page', 'N/A')}]\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )
    prompt = RAG_PROMPT.format(context=context, question=question)
    for attempt in range(retry + 1):
        try:
            resp = solar_client.chat.completions.create(
                model=SOLAR_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=512
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt == retry:
                return f"ERROR: {e}"
            time.sleep(3)


def save_checkpoint(records: list, path: str):
    rows = []
    for r in records:
        row = {
            "id": r["id"], "type": r["type"],
            "user_input": r["user_input"],
            "answer":     r["answer"],
            "reference":  r["reference"],
        }
        for i, ctx in enumerate(r["retrieved_contexts"]):
            row[f"ctx_{i}"] = ctx
        rows.append(row)
    pd.DataFrame(rows).to_csv(path, index=False, encoding='utf-8-sig')


# Retrieval
records = []
for _, row in df_gold.iterrows():
    docs = retriever.invoke(row['user_input'])
    records.append({
        "id":                 row['ID'],
        "type":               row['Type'],
        "user_input":         row['user_input'],
        "retrieved_contexts": [doc.page_content for doc in docs],
        "answer":             "",
        "reference":          row['reference'],
    })

print(f"\u2705 Retrieval 완료: {len(records)}개")
print(f"   컨텍스트 수 확인: {[len(r['retrieved_contexts']) for r in records[:3]]}")

In [ ]:
# Solar Answer 생성 (10개마다 체크포인트 저장)
start_idx = sum(1 for r in records if r['answer'] != "")
if start_idx > 0:
    print(f"\u23e9 {start_idx}개 완료 — {start_idx}번부터 이어서 실행")

start_time = time.time()

for i in tqdm(range(start_idx, len(records)), desc="Solar Answer 생성", initial=start_idx, total=len(records)):
    records[i]['answer'] = generate_answer_solar(records[i]['user_input'])
    if (i + 1) % 10 == 0:
        save_checkpoint(records, SAVE_RETRIEVED)
        elapsed = time.time() - start_time
        print(f"  \U0001f4be 체크포인트: {i+1}/100 ({elapsed:.0f}초 경과)")

save_checkpoint(records, SAVE_RETRIEVED)

elapsed     = time.time() - start_time
error_count = sum(1 for r in records if str(r['answer']).startswith('ERROR'))

print(f"\n\u2705 Solar Answer 생성 완료: {len(records)}개 ({elapsed:.0f}초)")
print(f"   저장: {SAVE_RETRIEVED}")
if error_count:
    print(f"   \u26a0\ufe0f  ERROR: {error_count}개 — RAGAS 평가 시 NaN으로 처리됩니다.")

## STEP 8 — (선택) 세션 재시작 시 중간 저장 파일 복원

> STEP 7이 이미 완료되었다면 이 셀은 **건너뛰세요**.

In [ ]:
import pandas as pd

df_retrieved = pd.read_csv(SAVE_RETRIEVED)
records = []
for _, row in df_retrieved.iterrows():
    contexts = [
        row[f"ctx_{i}"]
        for i in range(TOP_K)
        if pd.notna(row.get(f"ctx_{i}", "")) and row.get(f"ctx_{i}", "") != ""
    ]
    records.append({
        "id":                 row["id"],
        "type":               row["type"],
        "user_input":         row["user_input"],
        "retrieved_contexts": contexts,
        "answer":             row["answer"],
        "reference":          row["reference"],
    })

print(f"\u2705 중간 저장 파일 복원 완료: {len(records)}개")
error_count = sum(1 for r in records if str(r['answer']).startswith('ERROR'))
if error_count:
    print(f"   \u26a0\ufe0f  ERROR {error_count}개 포함 — STEP 7 재실행 권장")

## STEP 9 — RAGAS 평가 모델 설정 (Solar judge)

> - Judge LLM: `solar-pro3`
> - Faithfulness 임베딩: 사용 안 함 (LLM judge만 사용)
> - Answer Relevancy 임베딩: `jhgan/ko-sroberta-multitask` (STEP 5에서 로드한 객체 재사용)
>   - Solar embedding API가 RAGAS 입력 형식(`$.input`)을 거부하여 jhgan으로 대체

In [ ]:
import pandas as pd
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from ragas import EvaluationDataset

# Solar judge LLM
ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=SOLAR_MODEL,
        openai_api_key=UPSTAGE_API_KEY,
        openai_api_base=SOLAR_BASE_URL,
        temperature=0,
        max_tokens=512
    )
)

# Answer Relevancy 임베딩 — jhgan (STEP 5에서 로드한 객체 재사용)
# Solar embedding은 RAGAS $.input 형식 미지원으로 대체
ragas_embeddings_hf = LangchainEmbeddingsWrapper(embeddings)

run_config = RunConfig(max_workers=2, timeout=120, max_retries=3)

eval_dataset = EvaluationDataset.from_pandas(pd.DataFrame([{
    "user_input":         r['user_input'],
    "retrieved_contexts": r['retrieved_contexts'],
    "response":           r['answer'],
    "reference":          r['reference'],
} for r in records]))

print(f"\u2705 RAGAS 평가 모델 설정 완료")
print(f"   Judge LLM        : {SOLAR_MODEL}")
print(f"   AR 임베딩         : {EMBED_MODEL} (jhgan)")
print(f"   EvaluationDataset: {len(eval_dataset)}개")

## STEP 10 — Faithfulness 평가

> 예상 소요: 약 10~20분 (실측: 484초)

In [ ]:
from ragas import evaluate
from ragas.metrics import Faithfulness
import time

print(f"\u25b6 Faithfulness 평가 시작 (judge: {SOLAR_MODEL})")
start = time.time()

result_faith = evaluate(
    dataset=eval_dataset,
    metrics=[Faithfulness()],
    llm=ragas_llm,
    embeddings=ragas_embeddings_hf,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)
df_faith = result_faith.to_pandas()
df_faith['id']   = [r['id']   for r in records]
df_faith['type'] = [r['type'] for r in records]
df_faith.to_csv(SAVE_FAITHFULNESS, index=False, encoding='utf-8-sig')

elapsed    = time.time() - start
faith_mean = df_faith['faithfulness'].mean()
faith_nan  = df_faith['faithfulness'].isna().sum()

print(f"\n\u2705 Faithfulness 완료 ({elapsed:.0f}초)")
print(f"   Faithfulness : {faith_mean:.4f}  (NaN: {faith_nan}개)")
print(f"   llama 기준   : {LLAMA_RESULTS['전체']['faithfulness']:.4f}  "
      f"({'\u2191' if faith_mean > LLAMA_RESULTS['전체']['faithfulness'] else '\u2193'} "
      f"{faith_mean - LLAMA_RESULTS['전체']['faithfulness']:+.4f})")
print(f"   저장: {SAVE_FAITHFULNESS}")

## STEP 11 — Answer Relevancy 평가

> 예상 소요: 약 3~5분 (실측: 169초)
>
> ⚠️ Solar API 호환 이슈로 인한 설정:
> - `strictness=1`: Solar가 `n=1`만 지원 → 기본값 `strictness=3` (n=3) 사용 불가
> - `ragas_embeddings_hf`: Solar embedding `$.input` 형식 오류로 jhgan 사용

In [ ]:
from ragas.metrics import AnswerRelevancy

# strictness=1: Solar n=1 제약 우회 (기본값 3은 n=3 요청으로 Solar에서 400 오류)
metric_ar = AnswerRelevancy(strictness=1)

print(f"\u25b6 Answer Relevancy 평가 시작 (strictness=1, jhgan embedding)")
start = time.time()

result_rel = evaluate(
    dataset=eval_dataset,
    metrics=[metric_ar],
    llm=ragas_llm,
    embeddings=ragas_embeddings_hf,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)
df_rel = result_rel.to_pandas()
df_rel['id']   = [r['id']   for r in records]
df_rel['type'] = [r['type'] for r in records]
df_rel.to_csv(SAVE_RELEVANCY, index=False, encoding='utf-8-sig')

elapsed  = time.time() - start
rel_mean = df_rel['answer_relevancy'].mean()
rel_nan  = df_rel['answer_relevancy'].isna().sum()

print(f"\n\u2705 Answer Relevancy 완료 ({elapsed:.0f}초)")
print(f"   Answer Relevancy : {rel_mean:.4f}  (NaN: {rel_nan}개)")
print(f"   llama 기준       : {LLAMA_RESULTS['전체']['answer_relevancy']:.4f}")

## STEP 12 — 결과 병합 + 유형별 분석

In [ ]:
import numpy as np

METRICS = ['faithfulness', 'answer_relevancy']

df_raw = pd.DataFrame({
    'id':               [r['id']        for r in records],
    'type':             [r['type']       for r in records],
    'user_input':       [r['user_input'] for r in records],
    'faithfulness':     df_faith['faithfulness'].values,
    'answer_relevancy': df_rel['answer_relevancy'].values,
})

overall_mean = df_raw[METRICS].mean().round(4)
overall_nan  = df_raw[METRICS].isna().sum()
summary      = df_raw.groupby('type')[METRICS].mean().round(4)

print("전체 평균 (NaN 제외):")
for m in METRICS:
    print(f"  {m:<22}: {overall_mean[m]:.4f}  (NaN: {overall_nan[m]}개)")

print("\n유형별 평균:")
print(summary.to_string())

print("\n유형별 NaN:")
print(df_raw.groupby('type')[METRICS].apply(lambda x: x.isna().sum()).to_string())

## STEP 13 — Solar vs llama3.1:8b 비교

In [ ]:
types_order = ['사실 확인', '비교', '의견/예측', '범위 밖']

print("=" * 72)
print("  Faithfulness + Answer Relevancy: Solar vs llama3.1:8b")
print("=" * 72)

for metric in METRICS:
    print(f"\n[{metric}]")
    print(f"  {'유형':<10} {'Solar(이번)':>12} {'llama(기존)':>12} {'차이':>8}")
    print("  " + "-" * 46)
    for t in types_order:
        solar_val = summary.loc[t, metric] if t in summary.index else np.nan
        llama_val = LLAMA_RESULTS.get(t, {}).get(metric, np.nan)
        diff      = f"{solar_val - llama_val:+.4f}" if not (np.isnan(solar_val) or np.isnan(llama_val)) else "   N/A"
        print(f"  {t:<10} {solar_val:>12.4f} {llama_val:>12.4f} {diff:>8}")
    solar_all = overall_mean[metric]
    llama_all = LLAMA_RESULTS['전체'][metric]
    print("  " + "-" * 46)
    print(f"  {'전체':<10} {solar_all:>12.4f} {llama_all:>12.4f} {solar_all - llama_all:>+8.4f}")

## STEP 14 — 생존 편향 보정 + 최종 저장 + README 마크다운 출력

In [ ]:
# 생존 편향 보정
print("=" * 60)
print("  생존 편향 보정 (NaN → 0점, 전체 100개 기준)")
print("=" * 60)
print(f"  {'지표':<22} {'낙관적(NaN제외)':>15} {'유효샘플':>8} {'보수적(전체100개)':>18}")
print("  " + "-" * 65)
for m in METRICS:
    opt = overall_mean[m]
    vn  = 100 - overall_nan[m]
    pes = round(opt * vn / 100, 4)
    print(f"  {m:<22} {opt:>15.4f} {vn:>8}개 {pes:>18.4f}")

In [ ]:
# 최종 저장 (Context Recall/Precision은 LLM 무관 → 기존 수치 병합)
df_raw['context_recall']    = LLAMA_RESULTS['전체']['context_recall']
df_raw['context_precision'] = LLAMA_RESULTS['전체']['context_precision']
df_raw.to_csv(SAVE_RAW, index=False, encoding='utf-8-sig')

df_s = summary.copy()
df_s.loc['전체'] = overall_mean
df_s.to_csv(SAVE_SUMMARY, encoding='utf-8-sig')

print(f"\u2705 저장 완료")
print(f"  raw     → {SAVE_RAW}")
print(f"  summary → {SAVE_SUMMARY}")

In [ ]:
# README / IEP 문서용 마크다운 출력
ALL_MEANS = {
    'context_recall':    LLAMA_RESULTS['전체']['context_recall'],
    'context_precision': LLAMA_RESULTS['전체']['context_precision'],
    'faithfulness':      overall_mean['faithfulness'],
    'answer_relevancy':  overall_mean['answer_relevancy'],
}

md  = "| 유형 | Context Recall | Context Precision | Faithfulness | Answer Relevancy |\n"
md += "| :--- | :---: | :---: | :---: | :---: |\n"
for t in types_order:
    cr = LLAMA_RESULTS.get(t, {}).get('context_recall', float('nan'))
    cp = LLAMA_RESULTS.get(t, {}).get('context_precision', float('nan'))
    fa = summary.loc[t, 'faithfulness']     if t in summary.index else float('nan')
    ar = summary.loc[t, 'answer_relevancy'] if t in summary.index else float('nan')
    md += f"| {t} | {cr:.4f} | {cp:.4f} | {fa:.4f} | {ar:.4f} |\n"
md += (f"| **전체** | **{ALL_MEANS['context_recall']:.4f}** "
       f"| **{ALL_MEANS['context_precision']:.4f}** "
       f"| **{ALL_MEANS['faithfulness']:.4f}** "
       f"| **{ALL_MEANS['answer_relevancy']:.4f}** |\n")

print("[README 마크다운 — IEP-1001 CASE 3 (Solar judge 기준)]")
print(md)

nan_md  = "| 지표 | 낙관적 (NaN 제외) | 유효 샘플 | 보수적 (전체 100개) |\n"
nan_md += "| :--- | :---: | :---: | :---: |\n"
for m in METRICS:
    opt = overall_mean[m]
    vn  = 100 - overall_nan[m]
    pes = round(opt * vn / 100, 4)
    nan_md += f"| {m} | {opt:.4f} | {vn}개 | **{pes:.4f}** |\n"
print("[생존 편향 보정 마크다운]")
print(nan_md)

print("=" * 60)
print("IEP-1001 CASE3 RAGAS 재측정 완료 (Solar judge)")
print("=" * 60)
print(f"  Context Recall    : {ALL_MEANS['context_recall']:.4f}  (llama 기준, LLM 무관)")
print(f"  Context Precision : {ALL_MEANS['context_precision']:.4f}  (llama 기준, LLM 무관)")
print(f"  Faithfulness      : {ALL_MEANS['faithfulness']:.4f}  ← Solar 실측  (llama: {LLAMA_RESULTS['전체']['faithfulness']:.4f})")
print(f"  Answer Relevancy  : {ALL_MEANS['answer_relevancy']:.4f}  ← Solar 실측  (llama: {LLAMA_RESULTS['전체']['answer_relevancy']:.4f})")
print()
print("\u26a0\ufe0f  Answer Relevancy 주의사항")
print("   - strictness=1 적용 (Solar n=1 제약, 기본값 3 사용 불가)")
print("   - jhgan 임베딩 사용 (Solar embedding $.input 형식 오류로 대체)")
print("   - llama 수치(0.6143)와 직접 비교 불가")